# shopextract -- E-Commerce Product Intelligence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umerkhan95/shopextract/blob/main/notebooks/shopextract_demo.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/umerkhan95/shopextract/blob/main/notebooks/shopextract_demo.ipynb)

Extract, analyze, validate, and monitor product data from any online store.

In [ ]:
!pip install shopextract -q

import shopextract

print(f"shopextract v{shopextract.__version__}")

## Sample Data

Load sample products from GitHub for the offline demos (analysis, validation, export, etc.).
Detection and extraction cells below hit the network independently.

In [ ]:
import json, urllib.request

url = "https://raw.githubusercontent.com/umerkhan95/shopextract/main/notebooks/sample_data.json"
products_as_dicts = json.loads(urllib.request.urlopen(url).read())
print(f"Loaded {len(products_as_dicts)} products")

## Platform Detection

`detect()` probes a URL using HTTP headers, API endpoints, and HTML signals to identify the e-commerce platform.

In [ ]:
STORE_URL = input("Enter a store URL to detect (e.g. https://example-store.com): ").strip() or "https://example-store.com"

result = await shopextract.detect(STORE_URL)
print(f"Platform:   {result.platform.value}")
print(f"Confidence: {result.confidence:.0%}")
print(f"Signals:    {', '.join(result.signals)}")

## Product Extraction

`extract()` detects the platform, discovers product URLs, and extracts structured data using a tiered strategy (API > crawl > CSS).

In [ ]:
EXTRACT_URL = input("Enter a store URL to extract (e.g. https://example-store.com): ").strip() or "https://example-store.com"

result = await shopextract.extract(EXTRACT_URL, max_urls=5)
print(f"Platform:        {result.platform.value}")
print(f"Extraction Tier: {result.tier.value}")
print(f"Products Found:  {result.product_count}")

## Display Products

Show the sample product catalog as a table.

In [ ]:
import pandas as pd

df_display = pd.DataFrame([
    {
        "Title": p["title"][:50],
        "Price": f"${p['price']:.2f}" if isinstance(p["price"], (int, float)) else f"${p['price']}",
        "Vendor": p.get("vendor") or "--",
        "SKU": p.get("sku") or "--",
        "In Stock": p.get("in_stock", True),
    }
    for p in products_as_dicts
])
df_display

## Catalog Analysis

Aggregate statistics, price distributions, and brand breakdowns from product data.

In [ ]:
stats = shopextract.analyze_products(products_as_dicts)

print(f"Total Products:     {stats.total_products}")
print(f"Price Range:        ${stats.price_range[0]:.2f} -- ${stats.price_range[1]:.2f}")
print(f"Average Price:      ${stats.avg_price:.2f}")
print(f"Median Price:       ${stats.median_price:.2f}")
print(f"In Stock:           {stats.in_stock}")
print(f"Out of Stock:       {stats.out_of_stock}")
print(f"Has GTIN:           {stats.has_gtin}")
print(f"Completeness Score: {stats.completeness_score:.1%}")

In [ ]:
dist = shopextract.price_distribution(products_as_dicts)

print("Price Distribution")
print("=" * 35)
for bucket, count in dist.items():
    bar = "#" * count
    print(f"  ${bucket:>10s}  {count:>3d}  {bar}")

In [ ]:
brands = shopextract.brand_breakdown(products_as_dicts)

print("Brand Share of Catalog")
print("=" * 40)
for brand, pct in list(brands.items())[:10]:
    bar = "#" * int(pct / 2)
    print(f"  {brand:<25s} {pct:5.1f}%  {bar}")

## Product Matching

`fuzzy_match()` finds the same product across catalogs by title similarity. `match_gtin()` does exact barcode lookups.

In [ ]:
store_a = [
    {"title": "Classic Percale Sheet Set - White",  "price": "149.00", "currency": "USD", "gtin": "0850031234567"},
    {"title": "Luxe Sateen Duvet Cover - Cream",    "price": "199.00", "currency": "USD", "gtin": "0850031234890"},
    {"title": "Down Alternative Comforter",          "price": "249.00", "currency": "USD", "gtin": "0850031234111"},
    {"title": "Waffle Bath Towel Set",               "price": "79.00",  "currency": "USD"},
]
store_b = [
    {"title": "Classic Percale Sheet Set (White)",   "price": "139.00", "currency": "USD", "gtin": "0850031234567"},
    {"title": "Luxe Sateen Duvet Cover, Cream",      "price": "209.00", "currency": "USD"},
    {"title": "Organic Cotton Pillowcase Pair",      "price": "49.00",  "currency": "USD"},
    {"title": "Waffle Towel Bath Set",               "price": "69.00",  "currency": "USD"},
]

matches = shopextract.fuzzy_match(store_a, store_b, threshold=0.7)

print(f"Found {len(matches)} matching products across stores\n")
for prod_a, prod_b, score in matches:
    print(f"  {prod_a['title']:<40s}  {prod_b['title']:<40s}  {score:.1%}")

In [ ]:
all_products = store_a + store_b
gtin_matches = shopextract.match_gtin("0850031234567", all_products)

print(f"Products matching GTIN 0850031234567: {len(gtin_matches)}\n")
for m in gtin_matches:
    print(f"  {m['title']:<45s}  ${m['price']:>7s}  GTIN: {m['gtin']}")

## Marketplace Validation

`validate()` checks product data against marketplace requirements (mandatory fields, format rules, character limits).

In [ ]:
report = shopextract.validate(products_as_dicts, marketplace="google_shopping")

print(f"Google Shopping: {report.valid}/{report.total} valid ({report.pass_rate:.0f}% pass rate)\n")
for issue in report.issues[:10]:
    icon = "ERROR" if issue.severity == "error" else "WARN "
    print(f"  [{icon}] {issue.product_title}: {issue.field} -- {issue.error}")

In [ ]:
report = shopextract.validate(products_as_dicts, marketplace="idealo")

print(f"idealo: {report.valid}/{report.total} valid ({report.pass_rate:.0f}% pass rate)\n")
for issue in report.issues[:10]:
    icon = "ERROR" if issue.severity == "error" else "WARN "
    print(f"  [{icon}] {issue.product_title}: {issue.field} -- {issue.error}")

In [ ]:
report = shopextract.validate(products_as_dicts, marketplace="amazon")

print(f"Amazon: {report.valid}/{report.total} valid ({report.pass_rate:.0f}% pass rate)\n")
for issue in report.issues[:10]:
    icon = "ERROR" if issue.severity == "error" else "WARN "
    print(f"  [{icon}] {issue.product_title}: {issue.field} -- {issue.error}")

## Price Monitoring

Track product prices over time using SQLite snapshots. Detect price changes, new arrivals, and removed products.

In [ ]:
import json, sqlite3, tempfile, os
from datetime import datetime, timezone, timedelta

tmp_db = tempfile.NamedTemporaryFile(suffix=".db", delete=False)
db_path = tmp_db.name
tmp_db.close()

conn = sqlite3.connect(db_path)
conn.execute("""
    CREATE TABLE snapshots (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        domain TEXT NOT NULL,
        products_json TEXT NOT NULL,
        created_at TEXT NOT NULL
    )
""")

snap_old = [
    {"title": "Classic Percale Sheet Set", "price": "149.00", "currency": "USD"},
    {"title": "Luxe Sateen Duvet Cover",   "price": "199.00", "currency": "USD"},
    {"title": "Down Alternative Comforter", "price": "249.00", "currency": "USD"},
    {"title": "Waffle Bath Robe",           "price": "98.00",  "currency": "USD"},
]
snap_new = [
    {"title": "Classic Percale Sheet Set", "price": "129.00", "currency": "USD"},
    {"title": "Luxe Sateen Duvet Cover",   "price": "219.00", "currency": "USD"},
    {"title": "Down Alternative Comforter", "price": "249.00", "currency": "USD"},
    {"title": "Organic Cotton Towel Bundle","price": "89.00",  "currency": "USD"},
]

ts_old = (datetime.now(timezone.utc) - timedelta(days=7)).isoformat()
ts_new = datetime.now(timezone.utc).isoformat()
conn.execute("INSERT INTO snapshots (domain, products_json, created_at) VALUES (?,?,?)",
             ("demo.store", json.dumps(snap_old), ts_old))
conn.execute("INSERT INTO snapshots (domain, products_json, created_at) VALUES (?,?,?)",
             ("demo.store", json.dumps(snap_new), ts_new))
conn.commit()
conn.close()
print(f"Snapshots: {len(snap_old)} products ({ts_old[:10]}) -> {len(snap_new)} products ({ts_new[:10]})")

In [ ]:
detected = shopextract.changes("demo.store", db_path=db_path)

print(f"Detected {len(detected)} changes\n")
for c in detected:
    if c.change_type.value == "price_change":
        d = "dropped" if c.new_price < c.old_price else "increased"
        print(f"  PRICE {d.upper():>9s}  {c.title}  ${c.old_price} -> ${c.new_price}")
    elif c.change_type.value == "new_product":
        print(f"  NEW PRODUCT       {c.title}  ${c.price}")
    elif c.change_type.value == "removed_product":
        print(f"  REMOVED           {c.title}  last: ${c.last_price}")

In [ ]:
history = shopextract.price_history("demo.store", "Classic Percale Sheet Set", db_path=db_path)

print("Price History: Classic Percale Sheet Set")
print("=" * 40)
for ts, price in history:
    print(f"  {ts.strftime('%Y-%m-%d %H:%M')}   ${price:.2f}")

os.unlink(db_path)

## Export

Export product data as CSV, JSON, Google Shopping XML, or Pandas DataFrame.

In [ ]:
import tempfile

csv_path = tempfile.mktemp(suffix=".csv")
shopextract.to_csv(products_as_dicts, csv_path)

with open(csv_path) as f:
    lines = f.readlines()
print("CSV (first 5 lines):")
for line in lines[:5]:
    print(line.rstrip()[:120])

In [ ]:
import json as _json

json_path = tempfile.mktemp(suffix=".json")
shopextract.to_json(products_as_dicts, json_path, indent=2)

with open(json_path) as f:
    parsed = _json.loads(f.read())
print("JSON (first product):")
print(_json.dumps(parsed[0], indent=2))

In [ ]:
feed_path = tempfile.mktemp(suffix=".xml")
shopextract.to_feed(products_as_dicts, feed_path, format="google_shopping")

with open(feed_path) as f:
    lines = f.readlines()
print("Google Shopping XML (first 20 lines):")
for line in lines[:20]:
    print(line.rstrip())

In [ ]:
df = shopextract.to_dataframe(products_as_dicts)
df[["title", "price", "currency", "vendor", "sku", "gtin"]]

## Quality Scoring

`QualityScorer` evaluates how complete each product record is on a 0.0--1.0 scale (title, price, image, description, SKU).

In [ ]:
scorer = shopextract.QualityScorer()

print(f"{'Product':<42s}  {'Score':>5s}  Fields")
print("=" * 75)
for p in products_as_dicts:
    score = scorer.score_product(p)
    fields = [k for k in ["title", "price", "image_url", "description", "sku"] if p.get(k)]
    print(f"  {p.get('title', '')[:42]:<42s}  {score:>4.2f}  {', '.join(fields)}")

In [ ]:
batch_score = scorer.score_batch(products_as_dicts)
print(f"Batch Average Quality: {batch_score:.2f}")
print(f"Products Scored:       {len(products_as_dicts)}")

## Duplicate Detection

`find_duplicates()` identifies duplicate products by fuzzy title matching or exact GTIN/SKU lookup.

In [ ]:
catalog = [
    {"title": "Classic Percale Sheet Set - White",  "price": "149.00", "gtin": "0850031234567", "sku": "PERC-001"},
    {"title": "Luxe Sateen Duvet Cover - Cream",    "price": "199.00", "gtin": "0850031234890", "sku": "SATN-001"},
    {"title": "Classic Percale Sheet Set (White)",   "price": "149.00", "gtin": "0850031234567", "sku": "PERC-001-V2"},
    {"title": "Down Alternative Comforter",          "price": "249.00", "gtin": "0850031234111", "sku": "DOWN-001"},
    {"title": "Luxe Sateen Duvet - Cream",           "price": "209.00", "gtin": "0850031234999", "sku": "SATN-002"},
]

title_dupes = shopextract.find_duplicates(catalog, method="title", threshold=0.8)

print(f"Title-Based Duplicates: {len(title_dupes)} pairs\n")
for idx_a, idx_b, sim in title_dupes:
    print(f"  [{idx_a}] {catalog[idx_a]['title']}")
    print(f"  [{idx_b}] {catalog[idx_b]['title']}")
    print(f"       Similarity: {sim:.1%}\n")

In [ ]:
gtin_dupes = shopextract.find_duplicates(catalog, method="gtin")

print(f"GTIN-Based Duplicates: {len(gtin_dupes)} pairs\n")
for idx_a, idx_b, sim in gtin_dupes:
    print(f"  [{idx_a}] {catalog[idx_a]['title']}")
    print(f"  [{idx_b}] {catalog[idx_b]['title']}")
    print(f"       GTIN: {catalog[idx_a]['gtin']}  (exact match)\n")

---

## Summary

This notebook covered: platform detection, product extraction, catalog analysis, product matching, marketplace validation (Google Shopping / idealo / Amazon), price monitoring, multi-format export, quality scoring, and duplicate detection.

- Install: `pip install shopextract`
- Source: [github.com/umerkhan95/shopextract](https://github.com/umerkhan95/shopextract)
- Docs: [shopextract.readthedocs.io](https://shopextract.readthedocs.io)